# 🤖 ISOM 260: Build Your First AI Agent

**Session 4 — Building AI Products** | Suffolk University | Prof. Hasan Arslan

---

## What's an AI Agent?

In Sessions 2–3, you learned to **talk to AI** and make **API calls**. But those were one-shot interactions: you ask, AI answers.

An **AI Agent** is different. It's an AI that can:

1. 🧠 **Think** about what it needs to do
2. 🛠️ **Use tools** — search the web, do math, look up data, take actions
3. 🔄 **Loop** — check its work, decide if it needs more info, and keep going
4. 💾 **Remember** — maintain context across steps

### The Formula

```
AI Agent = LLM + Tools + Reasoning Loop + Memory
```

### Why This Matters for Business

- **Chatbot**: "What's our revenue?" → "I don't have access to that data."
- **AI Agent**: "What's our revenue?" → *queries database* → *calculates growth* → *compares to forecast* → "Revenue is $2.3M, up 12% QoQ, 3% above forecast."

That's the leap. Let's build it.

---

### 💡 Real-World Context: NanoClaw & OpenClaw

Right now (March 2026), the hottest trend in AI is personal AI agents. **OpenClaw** — an open-source agent platform — exploded to 150,000+ users. But it has 400,000 lines of code and serious security issues (one user had their entire inbox deleted!).

Enter **NanoClaw** — built by a single developer with Claude Code in one weekend. Just **500 lines of core code**. Andrej Karpathy called it "really interesting" because it "fits into both my head and that of AI agents."

The lesson? **Simplicity wins.** Today, you'll build a mini-agent that captures the same core concept: an LLM that can use tools.

---

## 🚀 Part 1: Setup (2 minutes)

Install the Anthropic SDK and set your API key.

In [ ]:
# Install the Anthropic Python SDK
!pip install -q anthropic

In [ ]:
# Set your API key
# Get yours at: https://console.anthropic.com
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in the left sidebar → Add secret named ANTHROPIC_API_KEY
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except:
    # Option 2: Paste directly (less secure, but works for class)
    os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"  # <-- Replace this
    print("⚠️ Using hardcoded API key. Consider using Colab Secrets instead.")

In [ ]:
# Verify connection
import anthropic

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say 'Agent ready!' in exactly 2 words."}]
)

print(response.content[0].text)
print(f"\n✅ Connection successful! Using model: claude-sonnet-4-5-20250929")

---

## 🧠 Part 2: Understanding Tool Use (The Key Concept)

Here's the core idea that transforms a chatbot into an agent:

**Without tools:**
```
You: "What's 7,394 × 8,261?"
Claude: "Let me calculate... approximately 61,097,834" (might be wrong!)
```

**With tools:**
```
You: "What's 7,394 × 8,261?"
Claude: *thinks* "I should use the calculator tool for precision"
       *calls calculator(7394, 8261, 'multiply')*
       *gets back 61,086,634*
Claude: "7,394 × 8,261 = 61,086,634" (✅ exact!)
```

### How It Works (The Agent Loop)

```
1. You send a message + tool definitions to Claude
2. Claude THINKS about whether it needs a tool
3. If yes: Claude says "I want to call [tool] with [inputs]"
   → Your code EXECUTES the tool
   → You send the result BACK to Claude
   → Claude uses the result to form its answer
4. If no: Claude just answers directly
```

This is the **exact same pattern** that powers NanoClaw, OpenClaw, Claude Code, and every AI agent in the world. Let's build it!

---

## 🛠️ Part 3: Your First Tool — The Calculator

Let's give Claude access to a calculator. We need:
1. A **tool definition** (tells Claude what the tool does)
2. A **tool function** (the actual Python code that runs)
3. An **agent loop** (handles the back-and-forth)

In [ ]:
# ============================================
# STEP 1: Define the tool for Claude
# ============================================
# This is like a job description — it tells Claude what the tool can do

calculator_tool = {
    "name": "calculator",
    "description": (
        "A precise mathematical calculator. Use this tool whenever you need to "
        "perform arithmetic calculations. It handles addition, subtraction, "
        "multiplication, division, and exponentiation with perfect accuracy. "
        "Always prefer this tool over mental math for any calculation."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The mathematical expression to evaluate, e.g. '2 + 2' or '(100 * 0.15) + 50'"
            }
        },
        "required": ["expression"]
    }
}

print("📝 Tool defined: calculator")
print(f"   Description: {calculator_tool['description'][:80]}...")

In [ ]:
# ============================================
# STEP 2: Create the actual tool function
# ============================================
# This is the real Python code that runs when Claude calls the tool

def calculator(expression: str) -> str:
    """
    Safely evaluate a mathematical expression.
    Returns the result as a string.
    """
    try:
        # Only allow safe math operations
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return f"Error: Expression contains invalid characters. Only numbers and +-*/.() are allowed."

        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

# Test it!
print("Testing calculator:")
print(f"  2 + 2 = {calculator('2 + 2')}")
print(f"  7394 * 8261 = {calculator('7394 * 8261')}")
print(f"  (100 * 0.15) + 50 = {calculator('(100 * 0.15) + 50')}")
print("\n✅ Calculator tool is working!")

In [ ]:
# ============================================
# STEP 3: Build the Agent Loop!
# ============================================
# This is the HEART of every AI agent — the loop that connects
# Claude's thinking to real-world tool execution

import json

def run_agent(user_message: str, tools: list, tool_functions: dict, verbose=True):
    """
    Run an AI agent that can use tools to answer questions.

    This is the same core pattern used by NanoClaw, OpenClaw,
    Claude Code, and every AI agent in production today.
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"🗣️  User: {user_message}")
        print(f"{'='*60}")

    messages = [{"role": "user", "content": user_message}]
    step = 0

    while True:
        step += 1

        # 📡 Send message to Claude (with tool definitions)
        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=1024,
            system="You are a helpful assistant. Use the available tools whenever they would help you give a more accurate answer. Always show your reasoning.",
            tools=tools,
            messages=messages
        )

        if verbose:
            print(f"\n🔄 Step {step} | Stop reason: {response.stop_reason}")

        # Check if Claude wants to use a tool
        if response.stop_reason == "tool_use":
            # Process ALL content blocks (Claude might think + use tools)
            tool_results = []

            for block in response.content:
                if block.type == "text":
                    if verbose:
                        print(f"   🧠 Thinking: {block.text[:150]}..." if len(block.text) > 150 else f"   🧠 Thinking: {block.text}")

                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input

                    if verbose:
                        print(f"   🛠️  Calling tool: {tool_name}")
                        print(f"      Input: {json.dumps(tool_input)}")

                    # 🚀 EXECUTE the tool!
                    if tool_name in tool_functions:
                        result = tool_functions[tool_name](**tool_input)
                    else:
                        result = f"Error: Unknown tool '{tool_name}'"

                    if verbose:
                        print(f"      Result: {result}")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            # Add Claude's response and tool results to conversation
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

        else:
            # Claude is done — extract final answer
            final_answer = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer += block.text

            if verbose:
                print(f"\n🤖 Agent Answer:\n{final_answer}")
                print(f"\n✅ Done in {step} step(s)")

            return final_answer

        # Safety: prevent infinite loops
        if step > 10:
            return "Error: Agent exceeded maximum steps."

print("✅ Agent loop defined! Let's test it.")

In [ ]:
# ============================================
# 🎯 TEST IT! — Calculator Agent in Action
# ============================================

# Register our tools
tools = [calculator_tool]
tool_functions = {"calculator": calculator}

# Test 1: Simple math
run_agent("What is 7,394 multiplied by 8,261?", tools, tool_functions)

In [ ]:
# Test 2: Business calculation (needs multiple steps!)
run_agent(
    "I'm opening a coffee shop. My monthly rent is $4,500, ingredients cost $2,800, "
    "and staff costs $6,200. If I sell 3,400 cups per month at $5.75 each, "
    "what's my monthly profit? And how many months until I recoup a $45,000 startup investment?",
    tools, tool_functions
)

In [ ]:
# Test 3: Watch Claude decide NOT to use the tool
run_agent(
    "What is the capital of France?",
    tools, tool_functions
)

### 🌟 Key Insight

Did you notice? Claude **decided on its own** when to use the calculator and when not to. That's the magic — the LLM acts as the **brain** that decides which tools to use, when, and how. You don't write if/else logic. Claude figures it out.

This is exactly what makes AI agents powerful: they **reason about** which actions to take.

---

## 📊 Part 4: Multi-Tool Agent (The Real Power)

One tool is cool. But real agents have **multiple tools** and decide which one(s) to use. Let's give our agent a toolkit:

1. 🧮 **Calculator** — math operations
2. 📈 **Stock Lookup** — get stock prices (simulated)
3. 🌐 **Company Info** — get company details (simulated)

With these three tools, Claude can answer complex business questions like:
> "Compare the market caps of Apple and Microsoft and tell me which is larger"

In [ ]:
# ============================================
# Define multiple tools + their functions
# ============================================

# Tool 2: Stock price lookup (simulated data for classroom use)
stock_tool = {
    "name": "get_stock_price",
    "description": (
        "Look up the current stock price and key metrics for a publicly traded company. "
        "Provide the stock ticker symbol (e.g., AAPL for Apple, MSFT for Microsoft). "
        "Returns current price, daily change, 52-week high/low, and shares outstanding."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "Stock ticker symbol, e.g. 'AAPL', 'MSFT', 'GOOGL'"
            }
        },
        "required": ["ticker"]
    }
}

def get_stock_price(ticker: str) -> str:
    """Simulated stock data for classroom use."""
    stocks = {
        "AAPL": {"name": "Apple Inc.", "price": 242.58, "change": "+1.23%", "high_52w": 260.10, "low_52w": 164.08, "shares_outstanding": "15.12B"},
        "MSFT": {"name": "Microsoft Corp.", "price": 428.50, "change": "-0.45%", "high_52w": 468.35, "low_52w": 388.46, "shares_outstanding": "7.43B"},
        "GOOGL": {"name": "Alphabet Inc.", "price": 182.30, "change": "+0.87%", "high_52w": 201.42, "low_52w": 150.22, "shares_outstanding": "12.20B"},
        "AMZN": {"name": "Amazon.com Inc.", "price": 215.45, "change": "+2.10%", "high_52w": 242.52, "low_52w": 166.48, "shares_outstanding": "10.52B"},
        "TSLA": {"name": "Tesla Inc.", "price": 342.10, "change": "-3.21%", "high_52w": 488.54, "low_52w": 138.80, "shares_outstanding": "3.21B"},
        "NVDA": {"name": "NVIDIA Corp.", "price": 138.25, "change": "+1.95%", "high_52w": 153.13, "low_52w": 75.61, "shares_outstanding": "24.49B"},
    }
    ticker = ticker.upper().strip()
    if ticker in stocks:
        return json.dumps(stocks[ticker])
    return json.dumps({"error": f"Ticker '{ticker}' not found. Available: {', '.join(stocks.keys())}"})


# Tool 3: Company info
company_tool = {
    "name": "get_company_info",
    "description": (
        "Get detailed information about a company including its sector, "
        "number of employees, founding year, CEO, and headquarters location. "
        "Use the stock ticker symbol to look up the company."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "Stock ticker symbol, e.g. 'AAPL'"
            }
        },
        "required": ["ticker"]
    }
}

def get_company_info(ticker: str) -> str:
    """Simulated company info for classroom use."""
    companies = {
        "AAPL": {"name": "Apple Inc.", "sector": "Technology", "employees": 164000, "founded": 1976, "ceo": "Tim Cook", "hq": "Cupertino, CA"},
        "MSFT": {"name": "Microsoft Corp.", "sector": "Technology", "employees": 228000, "founded": 1975, "ceo": "Satya Nadella", "hq": "Redmond, WA"},
        "GOOGL": {"name": "Alphabet Inc.", "sector": "Technology", "employees": 182000, "founded": 1998, "ceo": "Sundar Pichai", "hq": "Mountain View, CA"},
        "AMZN": {"name": "Amazon.com Inc.", "sector": "Consumer Cyclical", "employees": 1540000, "founded": 1994, "ceo": "Andy Jassy", "hq": "Seattle, WA"},
        "TSLA": {"name": "Tesla Inc.", "sector": "Automotive", "employees": 140000, "founded": 2003, "ceo": "Elon Musk", "hq": "Austin, TX"},
        "NVDA": {"name": "NVIDIA Corp.", "sector": "Technology", "employees": 32000, "founded": 1993, "ceo": "Jensen Huang", "hq": "Santa Clara, CA"},
    }
    ticker = ticker.upper().strip()
    if ticker in companies:
        return json.dumps(companies[ticker])
    return json.dumps({"error": f"Company '{ticker}' not found."})


# Register all tools
all_tools = [calculator_tool, stock_tool, company_tool]
all_functions = {
    "calculator": calculator,
    "get_stock_price": get_stock_price,
    "get_company_info": get_company_info
}

print("✅ 3 tools registered:")
for t in all_tools:
    print(f"   🛠️  {t['name']}")

In [ ]:
# 🎯 Multi-tool test: Claude needs to use MULTIPLE tools and REASON across them

run_agent(
    "Compare Apple and Microsoft: which company has a higher market cap? "
    "Also tell me which company has more revenue per employee if Apple's "
    "annual revenue is $385 billion and Microsoft's is $245 billion.",
    all_tools, all_functions
)

In [ ]:
# 🎯 Another complex query

run_agent(
    "I have $100,000 to invest. If I split it equally between NVIDIA and Tesla, "
    "how many shares of each could I buy at current prices? "
    "Which company is older, and what sectors are they in?",
    all_tools, all_functions
)

### 🌟 Key Insight

Watch the step count! Claude is making **multiple tool calls** in sequence, using the results from one to inform the next. It:

1. **Plans** what information it needs
2. **Calls** the right tools in the right order
3. **Synthesizes** the results into a coherent answer

That's an **agent**. Not a chatbot. An agent.

---

## 🎨 Part 5: Build Your Own Tool! (Challenge)

Now it's your turn. Create a **custom tool** that solves a real problem.

### Idea Starters:
- 🌦️ **Weather lookup** — get weather for a city (simulated)
- 📅 **Meeting scheduler** — check available time slots
- 📧 **Email drafter** — generate email drafts with specific formatting
- 🍴 **Calorie counter** — look up nutrition info for foods
- 📚 **Course catalog** — search Suffolk courses by topic
- 💰 **Tip calculator** — split bills among friends

### Template — Copy and modify:

In [ ]:
# ============================================
# 🎨 YOUR CUSTOM TOOL — Fill in the blanks!
# ============================================

# Step 1: Define your tool (the "job description" for Claude)
my_custom_tool = {
    "name": "your_tool_name",           # <-- Change this
    "description": (
        "Describe what your tool does. Be detailed! "  # <-- Change this
        "Tell Claude when to use it and what it returns."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "param1": {                    # <-- Change parameter name
                "type": "string",           # <-- string, number, boolean, etc.
                "description": "What this parameter means"  # <-- Change this
            }
            # Add more parameters here if needed
        },
        "required": ["param1"]             # <-- Which params are required?
    }
}

# Step 2: Write the actual function
def your_tool_name(param1: str) -> str:    # <-- Match the tool name!
    """
    Your tool logic here.
    Must return a string.
    """
    # Your code here!
    return f"Result for {param1}"


# Step 3: Register it with all the other tools
my_tools = [calculator_tool, stock_tool, company_tool, my_custom_tool]
my_functions = {
    "calculator": calculator,
    "get_stock_price": get_stock_price,
    "get_company_info": get_company_info,
    "your_tool_name": your_tool_name        # <-- Match the tool name!
}

# Step 4: Test it!
run_agent(
    "Your test question that would trigger the tool",   # <-- Change this
    my_tools, my_functions
)

---

## 💭 Part 6: Reflection — The Bigger Picture

### What You Just Built vs. What's in Production

| What You Built | What NanoClaw/OpenClaw Do |
|---|---|
| 3 simulated tools | Hundreds of real tools (email, calendar, web, files) |
| Single conversation | Persistent memory across sessions |
| Runs in Colab | Runs in isolated containers 24/7 |
| You type messages | Connected to WhatsApp, Slack, Discord |
| ~50 lines of agent code | NanoClaw: ~500 lines. OpenClaw: ~400,000 lines |

But the **core pattern is identical**:

```
1. Receive message
2. Send to LLM with tool definitions
3. If LLM wants to use a tool → execute it → send result back
4. Repeat until LLM has a final answer
5. Return answer to user
```

**You now understand the architecture of every AI agent in the world.** The complexity in production comes from:
- More tools (and real APIs instead of simulated data)
- Security (containers, permissions, sandboxing)
- Memory (databases, conversation history)
- Reliability (error handling, retries, timeouts)
- UX (WhatsApp integration, web interfaces)

### 💡 The Business Insight

The **most valuable skill** isn't coding the agent loop (that's ~50 lines). It's:
1. **Identifying which tools to build** — What data sources matter? What actions are valuable?
2. **Writing great tool descriptions** — The AI is only as good as its understanding of what each tool does
3. **Designing the user experience** — How do people interact with your agent? When should it ask for help?

These are business decisions, not engineering decisions. That's why **you** are perfectly positioned to build the next generation of AI agents.

---

## 🚀 What's Next?

- **Session 5**: Data & Analytics — The data that powers AI products
- **Session 8**: AI Agents & Automation — Deep dive into building production agents
- **Homework**: Add a 4th tool to this notebook that solves a real problem you care about

---

**ISOM 260: AI for Business** | Suffolk University | Session 4

🌐 [isom-260.vercel.app](https://isom-260.vercel.app)